# Academic Voice Research Assistant

A fully local voice assistant designed for students and researchers.

**Pipeline:** Microphone → Faster-Whisper (Speech-to-Text) → Ollama + Llama 3.2 → Piper TTS → Speaker

This notebook runs end-to-end on Google Colab (free tier supported).

**Use cases:**
- Explain research concepts in plain language
- Help refine paper ideas or abstracts
- Generate short technical explanations
- Practice presenting research ideas verbally
- Quick clarification of methods, architectures, or papers

---

## Setup Instructions

1. Runtime → Change runtime type → **T4 GPU** (recommended)
2. Run the cells in order
3. Allow microphone access when prompted by the browser
4. Speak naturally. The assistant will reply with voice.

## 1. Install Dependencies

In [ ]:
# System packages (zstd is required by the Ollama installer)
!apt-get update -qq
!apt-get install -y -qq portaudio19-dev espeak-ng libespeak-ng1 zstd > /dev/null

# Python packages
!pip install -q faster-whisper piper-tts gradio ollama soundfile numpy

# Install Ollama
!curl -fsSL https://ollama.com/install.sh | sh

print("Installation complete.")

## 2. Start Ollama Server and Download Model

In [ ]:
import subprocess
import threading
import time
import os

def start_ollama_server():
    os.environ["OLLAMA_HOST"] = "0.0.0.0:11434"
    subprocess.Popen(
        ["ollama", "serve"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL
    )

threading.Thread(target=start_ollama_server, daemon=True).start()
time.sleep(6)

# Recommended model for Colab free tier
!ollama pull llama3.2:3b

print("Ollama is ready.")

## 3. Download Piper Voice Model

In [ ]:
!python -m piper.download_voices en_US-lessac-medium
print("Piper voice model ready.")

## 4. Load Models and Define Assistant

In [ ]:
import gradio as gr
import numpy as np
import soundfile as sf
import tempfile
import wave
from faster_whisper import WhisperModel
import ollama
from piper import PiperVoice

# Load speech-to-text model
print("Loading Whisper model...")
device = "cuda" if os.path.exists("/usr/local/cuda") else "cpu"
whisper_model = WhisperModel(
    "base",
    device=device,
    compute_type="int8"
)

# Load text-to-speech model
print("Loading Piper voice...")
piper_voice = PiperVoice.load("en_US-lessac-medium.onnx")

SYSTEM_PROMPT = """You are an academic research assistant specializing in artificial intelligence and machine learning.
Your responses should be clear, accurate, and suitable for spoken conversation.

Guidelines:
- Keep answers concise (usually 2 to 5 sentences).
- Prefer plain language while remaining technically precise.
- When explaining concepts, use short analogies or concrete examples when helpful.
- If the question is ambiguous, ask one clarifying question.
- Do not use markdown, bullet points, or code blocks in your replies.
- Speak as if you are having a verbal discussion with a researcher or graduate student."""

print("Models loaded successfully.")

## 5. Core Functions

In [ ]:
def transcribe_audio(audio):
    """Convert recorded audio to text using Faster-Whisper."""
    if audio is None:
        return None

    sample_rate, audio_data = audio

    # Convert to mono if needed
    if audio_data.ndim > 1:
        audio_data = audio_data.mean(axis=1)

    audio_data = audio_data.astype(np.float32)

    # Write temporary WAV file
    with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as tmp:
        sf.write(tmp.name, audio_data, sample_rate)
        temp_path = tmp.name

    segments, _ = whisper_model.transcribe(temp_path, language="en")
    transcript = " ".join(segment.text for segment in segments).strip()

    os.unlink(temp_path)
    return transcript if transcript else None


def generate_reply(user_text, conversation_history):
    """Generate a spoken-style response using Ollama."""
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]

    # Keep recent context
    for user_msg, assistant_msg in conversation_history[-4:]:
        messages.append({"role": "user", "content": user_msg})
        messages.append({"role": "assistant", "content": assistant_msg})

    messages.append({"role": "user", "content": user_text})

    response = ollama.chat(
        model="llama3.2:3b",
        messages=messages
    )
    return response["message"]["content"].strip()


def synthesize_speech(text):
    """Convert text to speech using Piper and return path to WAV file."""
    with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as tmp:
        output_path = tmp.name

    with wave.open(output_path, "wb") as wav_file:
        piper_voice.synthesize(text, wav_file)

    return output_path


def process_voice_input(audio, history):
    """Full pipeline: audio → text → LLM → speech."""
    if history is None:
        history = []

    transcript = transcribe_audio(audio)

    if not transcript:
        return history, None, "No speech detected. Please try again."

    reply = generate_reply(transcript, history)
    speech_path = synthesize_speech(reply)

    history.append((transcript, reply))
    return history, speech_path, transcript

## 6. Launch the Interface

In [ ]:
with gr.Blocks(title="Academic Voice Research Assistant") as demo:
    gr.Markdown(
        """
        # Academic Voice Research Assistant

        Speak naturally about research topics, concepts, paper ideas, or technical questions.
        The assistant will reply using voice.

        **Recommended:** Use a quiet environment and speak clearly for 3–10 seconds.
        """
    )

    chatbot = gr.Chatbot(
        label="Conversation History",
        height=380
    )

    status_box = gr.Textbox(
        label="Last Transcription",
        interactive=False
    )

    with gr.Row():
        audio_input = gr.Audio(
            sources=["microphone"],
            type="numpy",
            label="Speak"
        )
        audio_output = gr.Audio(
            label="Assistant Reply",
            autoplay=True
        )

    clear_button = gr.Button("Clear Conversation")

    history_state = gr.State([])

    audio_input.stop_recording(
        fn=process_voice_input,
        inputs=[audio_input, history_state],
        outputs=[chatbot, audio_output, status_box]
    ).then(
        fn=lambda h: h,
        inputs=[history_state],
        outputs=[history_state]
    )

    clear_button.click(
        fn=lambda: ([], None, ""),
        outputs=[chatbot, audio_output, status_box]
    ).then(
        fn=lambda: [],
        outputs=[history_state]
    )

demo.launch(share=True, debug=False)

## Notes

- First run will take longer because models need to be downloaded.
- On free Colab, `llama3.2:3b` offers a good balance of speed and quality.
- You can change the model in the Ollama pull cell (for example `llama3.2:1b` for lower resource usage).
- Piper voice can be changed by downloading a different model in section 3.
- The assistant is intentionally constrained to short spoken answers for better conversation flow.